# 1.1. Domain Area

Dental radiographic images are one of the means of examining the condition of teeth and surrounding tissues. Unlike a standard visual examination, an X-ray image provides information about the internal structures of a tooth, its roots, and adjacent tissues, which may be partially or completely inaccessible to direct observation. Consequently, radiography can be used to identify and further analyse various clinically significant changes and conditions.

The project uses the DENTEX Challenge 2023 dataset, created for automated analysis of panoramic dental radiographs. The dataset can be examined in greater detail on the Kaggle platform: [DENTEX Challenge 2023](https://www.kaggle.com/datasets/truthisneverlinear/dentex-challenge-2023). DENTEX was initially created as a hierarchical benchmark: the task includes quadrant identification, tooth numbering according to the FDI system, and identification of the corresponding diagnostic category. For this purpose, the dataset provides different annotation levels: from quadrant-only data to fully annotated data with quadrant, tooth enumeration, and diagnosis.

This project does not reproduce the full original hierarchical DENTEX task. The main focus is detection by diagnosis categories: the model must locate the corresponding teeth or regions in a panoramic radiograph and distinguish among the four DENTEX categories:

- Caries;
- Deep Caries;
- Periapical Lesion;
- Impacted.

This class set creates a non-trivial task for the model. The categories are not four independent conditions at the same semantic level. In particular, Caries and Deep Caries describe related conditions, with the latter characterising a deeper carious lesion. In contrast, Periapical Lesion concerns a lesion in a different anatomical region, whereas Impacted characterises the eruption status and position of a tooth. Therefore, the model must not only locate visually distinct objects, but also distinguish closely related categories, primarily Caries and Deep Caries.

At the same time, the complexity of the task formulation is not a consequence of low annotation quality. DENTEX was created as a specialised medical benchmark, and the annotations underwent expert review: the initial annotation of each image was performed by a final-year dentistry student, after which it was reviewed and, where necessary, corrected by one of three experienced dentists with more than 15 years of professional experience. In addition, the images were acquired at three different medical institutions using different equipment and image acquisition protocols; therefore, the dataset contains a degree of variability characteristic of real clinical practice.

It is the combination of expert-reviewed annotations, real medical images, and a non-trivial class structure that makes this dataset suitable for studying computer vision methods. Within the project, the interest lies not only in the ability of a detector to locate the relevant region, but also in how well it can distinguish the represented categories and what types of errors arise for visually or semantically related classes.

Thus, the project domain is the automated analysis of panoramic dental radiographs for the purpose of locating and recognising clinically significant tooth conditions based on the diagnosis level of the DENTEX dataset.

# 1.2. ML Task Formulation

From a machine learning perspective, the project task is formulated as a multiclass object detection task. The model must determine the presence of objects of the specified classes in a panoramic dental radiograph and establish their locations.

A simpler alternative would be image classification. In that case, the model could determine which classes are present in a radiograph, but it would not indicate the locations of the corresponding objects. Such a result is insufficient for this task because one image contains many teeth and may contain several objects from different classes. It is necessary to determine not only the class, but also the image region to which the prediction corresponds.

For this purpose, object detection is used. The output of a detector is a set of detected objects. Each object is described by a bounding box, a class label, and a model confidence score. For an input image $I$, the model output can be written as

$$
f(I) = D = \{(b_i, c_i, s_i)\}_{i=1}^{N},
$$

where $D$ is the set of obtained detections, $N$ denotes their number, $c_i$ is the predicted class, and $s_i$ is the confidence score for the $i$-th object.

The bounding box is specified by the coordinates

$$
b_i = (x_{1i}, y_{1i}, x_{2i}, y_{2i}),
$$

where $(x_{1i}, y_{1i})$ and $(x_{2i}, y_{2i})$ define the positions of two opposite corners of the rectangular region.

The set of classes within the task has the form

$$
C = \{"Impacted", "Caries", "Periapical Lesion", "Deep Caries"\}.
$$

For each obtained detection,

$$
c_i \in C, \quad s_i \in [0, 1].
$$

Thus, the model solves two related subtasks. The first is object localisation in the image; the second is determining its class.

An alternative approach for working with such images could be image segmentation. In an object detection task, the object position is specified by a rectangular bounding box, so the model determines its approximate region. Segmentation operates at the level of individual pixels and makes it possible to determine the shape and boundaries of the required region more precisely. In the simplest case, the result can be represented by a binary mask

$$
M(x, y) \in \{0, 1\},
$$

where $M(x, y) = 1$ means that the pixel with coordinates $(x, y)$ belongs to the selected region, while $M(x, y) = 0$ corresponds to the background.

For the stated task, using segmentation is not necessary. The objective of the model is to identify a clinically significant object, determine its class, and localise it in the radiographic image. For this, it is sufficient to establish the region of its location using a bounding box. Precise determination of the object contour at the level of individual pixels is not part of the current task formulation.

Segmentation would be appropriate under a different formulation, where the geometric characteristics of the detected region themselves are the subject of analysis. For example, the task could be extended from detecting Caries to precisely determining the boundaries of a carious lesion. In that case, the model output would be a mask of the affected region, from which its area, shape, and extent relative to tooth tissues could be assessed. Similarly, for Periapical Lesion, segmentation could be used to determine the exact boundaries of the lesion in the periapical region.

Therefore, the choice between detection and segmentation is determined primarily by the task formulation. In this project, it is necessary to establish the presence of an object, its class, and its approximate spatial position; therefore, the task is formulated as object detection. Moving to analysis of the precise boundaries and geometry of detected regions would require formulating a segmentation task and providing corresponding pixel-level data annotations.

For model evaluation, mean Average Precision $\mathrm{mAP}@[0.5:0.95]$ was selected as the main metric. It accounts for the correctness of class determination and the quality of object localisation at different Intersection over Union (IoU) threshold values. Therefore, this metric is used as the main criterion when comparing architectures and selecting a model configuration.

Among Precision and Recall, Recall is assigned higher priority within the stated task. For a potential system supporting the analysis of radiographic images, missing a present clinically significant object, that is, a false negative, is a less desirable outcome than an additional false detection that can subsequently be checked by a specialist. Therefore, when $\mathrm{mAP}@[0.5:0.95]$ values are close, preference is given to the model with higher Recall. Precision is used for additional control of the number of false detections.

Thus, the primary model-quality criterion is $\mathrm{mAP}@[0.5:0.95]$, and among detection errors, greater attention is given to reducing the number of false negatives and increasing Recall. This set of criteria corresponds to the stated task because it accounts for classification quality, localisation accuracy, and the ability of the model to detect existing objects.

# 1.3. Project Goal and General Methodology

The goal of the project is to develop a complete process for building a computer vision system for automated detection of specified categories in panoramic dental X-ray images. The primary focus is not only on training a model and obtaining high metric values, but on completing the main stages of ML-system development: from working with initial data to evaluation, deployment, and monitoring of the completed model.

The methodological foundation of the project is CRISP-DM (Cross-Industry Standard Process for Data Mining). This approach divides the task workflow into the stages of Business Understanding, Data Understanding, Data Preparation, Modeling, Evaluation, and Deployment. Within the project, this structure was adapted to a computer vision task and supplemented with stages related to experimental model comparison, error analysis, and MLOps.

The initial stage of collecting and manually annotating medical images was not performed within the project. Creating a new dataset of dental radiographs is a separate complex task. It requires access to radiographic equipment and medical institutions, obtaining the necessary permissions to use medical data, ensuring patient confidentiality, and involving specialists for subsequent annotation review. For an object detection task, it is also necessary to localise each object in an image, for example by manually creating bounding boxes in specialised annotation tools such as CVAT. For a sufficiently large number of images, this process requires substantial time, organisational, and financial resources.

Therefore, the ready-made DENTEX dataset with existing expert annotations was selected as the starting point. The data-related project stages begin with validation, selection, and preparation of the available images and annotations, followed by exploratory data analysis and construction of a preprocessing pipeline.

The experimental part begins with constructing a baseline model, which is used as the initial point for subsequent comparisons. This is followed by training and comparing several detection architectures under a common protocol. Based on the results of this stage, the most promising model is selected for additional experiments with the training configuration, augmentations, input image resolution, and other parameters.

After model selection is completed, its final configuration is evaluated on a separate test set that is not used for model tuning. In addition to overall metrics, results for individual classes and characteristic errors are analysed: false positives, false negatives, classification errors, and localisation errors. This makes it possible to assess not only the final numerical quality of the model, but also the characteristics of its behaviour on different object types.

A separate project objective is the practical use of modern ML-system development tools. After the experimental part is completed, the model moves to the deployment stage: it is exported, an inference service and API are created, prediction results are persisted, and containerisation, testing, and collection of system-operation metrics are implemented. MLOps tools are used for versioning data and models, tracking experiments, automating individual processes, and monitoring.

The overall workflow sequence can be represented compactly as:

Task formulation $\rightarrow$ Data preparation $\rightarrow$ EDA $\rightarrow$ Preprocessing $\rightarrow$ Baseline $\rightarrow$ Architecture comparison $\rightarrow$ Refinement $\rightarrow$ Final evaluation $\rightarrow$ Error analysis $\rightarrow$ Deployment $\rightarrow$ Monitoring/MLOps

As a result, the project covers the main stages of the ML-system lifecycle. This approach makes it possible to consider the trained model not as an isolated experimental result, but as the central component of a system that also requires experiment reproducibility, independent evaluation, a software interface, persistence of results, and operational control after deployment.